# NB13：代糖混合物模型 — PLS vs 古典最小平方（NNLS/CLS）

**食品分析實驗 週次 16｜銜接 W15（代糖前處理）與 W16 課堂 deck**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 檢查 **5×5 全因子設計**（阿斯巴甜 0–10% × 三氯蔗糖 0–10% w/w，赤藻糖醇補足）的正交性（`corr(Y1,Y2)≈0`）。
2. 用**分組交叉驗證**（`GroupKFold`／`LeaveOneGroupOut`，以 `sample_id` 分組）比較前處理，並用 **PLS1** 分別對兩種代糖建模。
3. 用**離網格盲樣（off-grid blind blends）**算出誠實的 **RMSEP**，並整理成 model report card。
4. 算出 **VIP** 並對照拉曼位移的官能基指認，理解**封閉性（closure）**問題。
5. 用**純物質參考光譜**做 **NNLS（非負最小平方，古典最小平方 CLS）**擬合，跟 PLS 比較優缺點。
6. 用低濃度端資料估計**近似 LOD**（教學簡化法，非正式方法確效）。
7. 用 **Hotelling T²／Q 殘差** 判斷一個市售樣品是否落在模型的適用範圍內，並誠實報告結果。

## 如何使用本筆記本

- 直接「全部執行」即可用內建的**模擬 Raman 785 光譜**跑完整份筆記本。
- 資料格式與共用函式（`load_spectra`／`split_spectra`／`plot_spectra`）與 NB09–NB12 一致，規範見下方 Setup 小節。

> ⚠️ 本筆記本的資料是**模擬教學資料**，不是真實量測值。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import nnls
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneGroupOut, KFold

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 1607
np.random.seed(RNG_SEED)

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
# ---- Embedded demo spectra (fallback) ---------------------------------
# Exact content of data/nb13_demo_spectra.csv.
SAMPLE_CSV = """sample_id,group,instrument,rep,role,split,aspartame_pct,sucralose_pct,erythritol_pct,400,420,440,460,480,500,520,540,560,580,600,620,640,660,680,700,720,740,760,780,800,820,840,860,880,900,920,940,960,980,1000,1020,1040,1060,1080,1100,1120,1140,1160,1180,1200,1220,1240,1260,1280,1300,1320,1340,1360,1380,1400,1420,1440,1460,1480,1500,1520,1540,1560,1580,1600,1620,1640,1660,1680,1700,1720,1740,1760,1780,1800,class
CAL-00,0,sim_Raman785_demo,1,mixture,calibration,0.0,0.0,100.0,0.1108575,0.11228733,0.11068913,0.11636199,0.10401121,0.11141208,0.10710552,0.10985263,0.11569105,0.11199847,0.12465913,0.11670051,0.11174776,0.10607293,0.10862627,0.11276856,0.11133669,0.10918367,0.11206186,0.11380713,0.10528179,0.13783301,0.2840144,0.57940805,0.75524884,0.57831776,0.27720807,0.13790478,0.11573533,0.13107095,0.26030885,0.55100296,0.85104505,0.8549862,0.55420688,0.26038132,0.14062773,0.11590429,0.11038513,0.10950704,0.11012585,0.14972357,0.23503354,0.41200712,0.56146755,0.55747346,0.41397922,0.23109479,0.14684422,0.1144345,0.11603327,0.11490687,0.11625639,0.11467509,0.10108801,0.11324249,0.11614725,0.11919302,0.10335052,0.12054524,0.11931507,0.10697463,0.11399299,0.10037138,0.11679238,0.09963253,0.12089496,0.11555119,0.11403059,0.10841038,0.10442286,
CAL-00,0,sim_Raman785_demo,2,mixture,calibration,0.0,0.0,100.0,0.08569063,0.08310296,0.0747718,0.09145017,0.09239129,0.09293813,0.09128583,0.0916819,0.09090781,0.08327211,0.09074318,0.09713304,0.08348682,0.09117398,0.08503025,0.09144463,0.08279392,0.09244747,0.08705226,0.07928698,0.09217528,0.11758014,0.25717717,0.57191734,0.72951471,0.56164571,0.26142948,0.11189873,0.08906459,0.12317418,0.22959322,0.52736297,0.83320417,0.83492188,0.53976011,0.23148525,0.11660605,0.09935027,0.089693,0.08759685,0.08414085,0.12085444,0.21726231,0.38880764,0.54135861,0.54244608,0.38849893,0.21529309,0.11279504,0.0899637,0.08435978,0.08528202,0.10194648,0.08410388,0.08435477,0.08896272,0.08256043,0.09080081,0.08038859,0.09694429,0.09173666,0.08367238,0.08460756,0.08259824,0.08394318,0.08178906,0.0988784,0.08436719,0.08602323,0.08538842,0.08424441,
CAL-01,0,sim_Raman785_demo,1,mixture,calibration,0.0,2.5,97.5,0.10228304,0.10584332,0.11194734,0.11102353,0.09675696,0.10362371,0.0984125,0.0955528,0.097667,0.09624997,0.10491448,0.09639755,0.13109564,0.12849805,0.11405287,0.09381044,0.09816597,0.10106255,0.10742272,0.1212374,0.1241431,0.12826101,0.26666007,0.55464176,0.72519292,0.55939513,0.26632069,0.13662517,0.11182846,0.13575052,0.24055879,0.53387185,0.8320158,0.83991353,0.54007732,0.24945532,0.12776643,0.10015692,0.09401047,0.09806514,0.10311003,0.12720074,0.2276664,0.39018892,0.54626067,0.54349816,0.39424132,0.21802896,0.13325325,0.10447304,0.10311792,0.10183997,0.08996452,0.11127834,0.10392787,0.1031998,0.09221828,0.10487641,0.08896935,0.09690198,0.09908235,0.10568409,0.09716236,0.10030444,0.10401457,0.10285237,0.10214563,0.10010549,0.10463787,0.10200295,0.10516622,
CAL-01,0,sim_Raman785_demo,2,mixture,calibration,0.0,2.5,97.5,0.0988914,0.09399327,0.10817378,0.09842486,0.09891461,0.09161854,0.08728386,0.0904445,0.09348039,0.0982014,0.09868544,0.10287849,0.11175792,0.1205715,0.10412463,0.08521036,0.09322579,0.09821282,0.10283339,0.11291348,0.10477519,0.12930051,0.26472806,0.55823236,0.73031749,0.56286549,0.26045643,0.12261166,0.09029261,0.12121824,0.23401775,0.52233146,0.83058532,0.82916455,0.53097335,0.23350138,0.11168158,0.10184369,0.10102084,0.09695324,0.09686569,0.12911729,0.21349485,0.37943762,0.53776591,0.53040854,0.38473513,0.21221313,0.12662138,0.09609641,0.09118873,0.0929348,0.08731199,0.09592317,0.07847722,0.0952566,0.10031992,0.1014748,0.09711926,0.08572703,0.09598416,0.08838913,0.08974538,0.09600208,0.09191121,0.09810705,0.09226805,0.09744927,0.09009684,0.0946482,0.10623312,
CAL-02,0,sim_Raman785_demo,1,mixture,calibration,0.0,5.0,95.0,0.12211106,0.11451709,0.1215537,0.12121484,0.11918768,0.11845503,0.1109945,0.10903937,0.12294844,0.10215549,0.10540279,0.13082387,0.15944216,0.16305462,0.12783236,0.11870343,0.11105337,0.10655978,0.13364759,0.15878341,0.13029235,0.14177698,0.28248792,0.55753516,0.71821529,0.57373929,0.29798788,0.14055211,0.12001105,0.13839224,0.256129,0.5317932,0.83188543,0.84258008,0.55793737,0.25524569,0.13684574,0.1321899,0.10990571,0.11816074,0.12409935,0.1513984,0.2427909,0.39828046,0.54751767,0.53789351,0.40503992,0.22450633,0.13710296,0.11856319,0.11305734,0.123532,0.11521176,0.11867703,0.12395067,0.12175989,0.11508557,0.11162879,0.11681491,0.11594411,0.1222056,0.11786079,0.12071759,0.10676936,0.10733336,0.11894358,0.11177747,0.11330159,0.11686077,0.10944521,0.114281,
CAL-02,0,sim_Raman785_demo,2,mixture,calibration,0.0,5.0,95.0,0.06998949,0.05397598,0.06113963,0.06168544,0.06336167,0.04901705,0.06500227,0.06973216,0.05859919,0.05282359,0.06390412,0.07877435,0.11222147,0.11400478,0.07840534,0.06929645,0.06085872,0.06281492,0.07123836,0.09640423,0.08391632,0.09462891,0.2275337,0.50505344,0.66745689,0.51450355,0.22947714,0.10017179,0.06740825,0.08267664,0.19824446,0.49163352,0.77517469,0.78199944,0.50430673,0.20758195,0.08167974,0.06793262,0.04957797,0.06470373,0.07376941,0.10199032,0.18207011,0.34164944,0.49695919,0.49354275,0.34338805,0.18358573,0.0928052,0.06536405,0.05906965,0.05911302,0.06057756,0.05624296,0.05874325,0.05006703,0.05758969,0.05769511,0.05169876,0.06672663,0.05645974,0.06045573,0.06048233,0.05912058,0.06311506,0.05540181,0.06705063,0.06250463,0.06778961,0.05919871,0.069928,
CAL-03,0,sim_Raman785_demo,1,mixture,calibration,0.0,7.5,92.5,0.10948785,0.12941753,0.11490421,0.1150768,0.11840951,0.11563192,0.12503059,0.1127024,0.12058692,0.11370538,0.11944853,0.13645102,0.18580876,0.19231033,0.13190703,0.1153713,0.1272733,0.12315192,0.13552526,0.18426907,0.14357283,0.14785855,0.27299257,0.55290073,0.70802466,0.553702,0.27864838,0.14943026,0.122363,0.14307995,0.24680596,0.54252452,0.81951904,0.82940123,0.55968456,0.26281218,0.14333836,0.11296152,0.1163823,0.12349232,0.12683121,0.14798505,0.24304232,0.39716433,0.53716152,0.53741812,0.38946336,0.2275151,0.14548396,0.11905348,0.1215977,0.11436084,0.13135466,0.11177306,0.12017602,0.12282722,0.10924784,0.11708667,0.12124284,0.1080528,0.12060342,0.10882637,0.104444,0.1164897,0.10810082,0.12928704,0.1183911,0.11390713,0.11942631,0.10996504,0.11666223,
CAL-03,0,sim_Raman785_demo,2,mixture,calibration,0.0,7.5,92.5,0.10334126,0.10224207,0.09780718,0.10656968,0.1059655,0.09243554,0.09569311,0.10130011,0.09501654,0.10827691,0.101105,0.12436101,0.17268625,0.17924137,0.12196617,0.10471007,0.12114222,0.10375854,0.14225124,0.17128653,0.13661983,0.12729754,0.27066933,0.53881765,0.70222388,0.54251579,0.26149232,0.12129297,0.10239512,0.12265199,0.22938894,0.51350179,0.80741518,0.81342241,0.54092114,0.25668592,0.12284819,0.10265724,0.10535425,0.11578941,0.10952079,0.13856312,0.21975813,0.37437145,0.51485745,0.52388659,0.36795651,0.21706709,0.12959397,0.11033108,0.10818942,0.10713553,0.11186559,0.09130226,0.10845917,0.09341674,0.10338703,0.11011952,0.10899811,0.10178475,0.09866425,0.10302544,0.10127261,0.09681282,0.11273154,0.10359053,0.10486427,0.11232895,0.10700167,0.09215585,0.09060569,
CAL-04,0,sim_Raman785_demo,1,mixture,calibration,0.0,10.0,90.0,0.06598845,0.06853817,0.07527125,0.0657906,0.06498142,0.07571238,0.07702241,0.06777511,0.0737997,0.08253463,0.07024024,0.10761327,0.17250397,0.17524872,0.10434597,0.08781075,0.06810359,0.07634266,0.12123751,0.16586792,0.10792062,0.11015388,0.23488816,0.50778006,0.67077925,0.50338987,0.22743032,0.09797224,0.08812395,0.08805626,0.204018,0.46688051,0.77426971,0.78688674,0.52025325,0.23545145,0.11043177,0.08400762,0.05974651,0.07048606,0.0820879,0.10307373,0.19053237,0.34840462,0.47434812,0.48402735,0.33807612,0.18613771,0.09805384,0.07407235,0.08715938,0.08013626,0.0768608,0.07296643,0.07192892,0.07548021,0.08719126,0.0644303,0.07856153,0.08577816,0.08311951,0.07640345,0.07361004,0.07689526,0.0659252,0.07211915,0.07463912,0.07805543,0.08274777,0.08179208,0.07693629,
CAL-04,0,sim_Raman785_demo,2,mixture,calibration,0.0,10.0,90.0,0.10932374,0.09997891,0.1002647,0.10180414,0.10200433,0.10241547,0.10710039,0.09462922,0.09632889,0.088694,0.09561682,0.12058966,0.19004397,0.20094528,0.13822812,0.10231058,0.09869982,0.100298,0.14162576,0.19341563,0.13959978,0.13077264,0.25034179,0.53303727,0.68893427,0.52901508,0.26310596,0.11936379,0.11272792,0.12999977,0.23057299,0.49396149,0.79248757,0.79934746,0.53356117,0.26142802,0.1226477,0.11507927,0.09836814,0.09516867,0.10563654,0.130162,0.21459182,0.35577782,0.50009866,0.49938337,0.36912405,0.21056564,0.1372828,0.10901883,0.11251312,0.09467605,0.08930945,0.10681464,0.10515238,0.1057937,0.10275065,0.09437382,0.09383724,0.10501508,0.09466698,0.09270939,0.10735429,0.09003528,0.09252288,0.11022675,0.1020147,0.09199612,0.09831095,0.10230222,0.10458488,
CAL-05,0,sim_Raman785_demo,1,mixture,calibration,2.5,0.0,97.5,0.10401949,0.08743212,0.09798001,0.08899841,0.10383708,0.08921138,0.091239,0.1034711,0.09403792,0.09962603,0.10391979,0.10445187,0.10774648,0.09624549,0.11699436,0.09892923,0.09982995,0.09839143,0.09551885,0.10426869,0.11007037,0.13231402,0.2679669,0.56021273,0.72228391,0.55744223,0.26539828,0.12911914,0.10181344,0.128801,0.26778101,0.53757909,0.82385436,0.82855765,0.53473246,0.23995581,0.11950351,0.09117179,0.09484031,0.08937599,0.09842287,0.14126537,0.22500652,0.38814955,0.54485802,0.53803489,0.39032924,0.22027806,0.12443725,0.10482746,0.10682021,0.0940563,0.09229483,0.09097159,0.09371743,0.10181059,0.09065353,0.09231077,0.09556936,0.10205547,0.10463147,0.11229936,0.09916895,0.09709656,0.08535384,0.09833705,0.09458218,0.11116795,0.10066318,0.09163467,0.0943795,
CAL-05,0,sim_Raman785_demo,2,mixture,calibration,2.5,0.0,97.5,0.10796042,0.11578595,0.11768334,0.11302569,0.1147618,0.11661195,0.11241444,0.11509002,0.11521449,0.11083917,0.11710549,0.10901109,0.11062839,0.11229884,0.118544,0.11843716,0.11771123,0.10675384,0.12188718,0.11221828,0.11927667,0.13765019,0.27484029,0.5773071,0.74386122,0.57739322,0.28710263,0.14334352,0.11696396,0.14601293,0.27266599,0.54977348,0.84161967,0.83467063,0.55193332,0.25410221,0.12974004,0.11205209,0.10861056,0.11390995,0.11809295,0.14349106,0.24063395,0.40582602,0.56651148,0.55884118,0.41239214,0.23713521,0.15207358,0.11340001,0.10669675,0.11196518,0.10557541,0.12031424,0.11856881,0.11022944,0.11707735,0.10491386,0.11305232,0.11215125,0.11398054,0.11494572,0.10570662,0.1093983,0.11255179,0.11396145,0.10839585,0.10705313,0.11664919,0.11975761,0.10924423,
CAL-06,0,sim_Raman785_demo,1,mixture,calibration,2.5,2.5,95.0,0.05883712,0.06475486,0.07286256,0.0455381,0.06216085,0.07404228,0.06429197,0.05812241,0.06115007,0.05284716,0.06871508,0.06634647,0.0962041,0.08582624,0.06991776,0.06946617,0.06507865,0.07224654,0.08444434,0.09011819,0.07212846,0.094793,0.22663594,0.51007377,0.67656104,0.51488792,0.23421869,0.0951688,0.06842516,0.08843177,0.21670399,0.48877233,0.78674924,0.78904912,0.50642522,0.19254959,0.08194585,0.06145718,0.08076518,0.05651293,0.0590437,0.08842825,0.18843166,0.35966207,0.50216544,0.4959381,0.36120705,0.18782617,0.10505346,0.07092301,0.06768773,0.06068594,0.06257279,0.06782733,0.05517425,0.07118017,0.06692802,0.05977736,0.07416652,0.07436267,0.06933906,0.0755492,0.06818243,0.06327252,0.0521336,0.06264467,0.06601241,0.06688622,0.05040646,0.06089309,0.06035822,
CAL-06,0,sim_Raman785_demo,2,mixture,calibration,2.5,2.5,95.0,0.11041653,0.11960183,0.10958625,0.11259934,0.11140564,0.11458032,0.11861089,0.1094477,0.13021035,0.11196562,0.11016882,0.11144406,0.13808369,0.13896435,0.12634136,0.10909888,0.11953207,0.115174,0.11946089,0.13460668,0.13506122,0.14428925,0.27973165,0.56852278,0.71951841,0.57286375,0.27766949,0.14498685,0.12561049,0.13389628,0.26282974,0.55173805,0.82677287,0.83633967,0.53997027,0.24997754,0.14405466,0.11008193,0.12210584,0.11390629,0.13090047,0.14759302,0.23715576,0.40124158,0.54611111,0.53389518,0.40388143,0.23064538,0.14112989,0.11749822,0.11117848,0.10871141,0.11078583,0.12110248,0.11775721,0.11593029,0.12130746,0.1028148,0.10894637,0.10777157,0.11123043,0.13146605,0.13266398,0.12103744,0.10142013,0.10686908,0.12050515,0.11632347,0.12138828,0.11551239,0.11130522,
CAL-07,0,sim_Raman785_demo,1,mixture,calibration,2.5,5.0,92.5,0.08747466,0.09541223,0.08920719,0.09765353,0.07869394,0.08766226,0.09048517,0.0963823,0.09181524,0.09001323,0.08604899,0.10420045,0.13551049,0.14234128,0.10502192,0.08927492,0.08707194,0.0949758,0.10890469,0.13433918,0.11993584,0.12208435,0.25082532,0.52491811,0.69387192,0.52141687,0.26859084,0.12529628,0.09142673,0.12231167,0.24672684,0.50256911,0.7918457,0.81067205,0.52531847,0.24022988,0.12249707,0.08392277,0.09352178,0.0960077,0.1031839,0.12493857,0.21257592,0.35463302,0.50575969,0.51572441,0.37100394,0.2081884,0.11695987,0.09687007,0.07860452,0.09678668,0.08132049,0.09536955,0.09632119,0.09334051,0.08854935,0.09593188,0.08145167,0.08882572,0.09055724,0.09816303,0.09020871,0.09501967,0.09180452,0.09703836,0.08996998,0.08989947,0.09141103,0.08714162,0.0853926,
CAL-07,0,sim_Raman785_demo,2,mixture,calibration,2.5,5.0,92.5,0.11184723,0.10410458,0.09028693,0.09985374,0.09987979,0.10171543,0.09832322,0.10153906,0.09461338,0.10179187,0.0944178,0.11041148,0.14130454,0.14731876,0.10916392,0.09825845,0.09872817,0.10508269,0.10336866,0.14179669,0.1091884,0.12550945,0.25654013,0.53809142,0.7065129,0.53986449,0.25949247,0.12374084,0.10488928,0.12552831,0.25787245,0.51951729,0.80677415,0.80360141,0.5364665,0.24119988,0.12642891,0.10162024,0.10167199,0.10334171,0.12057248,0.13160392,0.22787184,0.38235144,0.5351304,0.5160088,0.37998808,0.21706019,0.13029706,0.10479915,0.11778924,0.10362817,0.10081051,0.09003471,0.0933513,0.10456937,0.107691,0.10311075,0.10029471,0.10803471,0.11335975,0.09540822,0.09896636,0.09984466,0.10362168,0.08510692,0.0970675,0.08678755,0.10369268,0.0966829,0.11217986,
CAL-08,0,sim_Raman785_demo,1,mixture,calibration,2.5,7.5,90.0,0.07552755,0.06561853,0.06729219,0.06710258,0.07331346,0.06755619,0.0754126,0.07168925,0.06748844,0.06831227,0.07161017,0.09209305,0.13739339,0.14086334,0.0859154,0.06422254,0.07296374,0.07205968,0.10458078,0.12530329,0.10373756,0.10599902,0.22398583,0.49613751,0.66329062,0.50965709,0.23198118,0.08835493,0.07940941,0.09734395,0.22196856,0.475634,0.74797453,0.75444834,0.50220898,0.21269807,0.10374256,0.07035172,0.0679612,0.07400598,0.07339778,0.1077673,0.18104945,0.33914273,0.4748594,0.48744819,0.34425228,0.18890162,0.10155807,0.07503156,0.08057105,0.06788129,0.07154857,0.07109843,0.0732077,0.05912252,0.07117596,0.06747054,0.06916893,0.07088237,0.07432073,0.08500887,0.06266704,0.06379553,0.06348678,0.07624084,0.0712778,0.07022021,0.06857912,0.06537659,0.0759995,
CAL-08,0,sim_Raman785_demo,2,mixture,calibration,2.5,7.5,90.0,0.09647833,0.10597435,0.09445595,0.10407274,0.10041916,0.10576222,0.10164429,0.10646213,0.10614716,0.09239637,0.08997929,0.12064068,0.16305457,0.15811189,0.12803378,0.09143446,0.10696306,0.10652789,0.12908209,0.16092789,0.13552057,0.12274963,0.25489669,0.50981763,0.68706125,0.53359874,0.25992802,0.136765,0.10147387,0.12179699,0.25929023,0.50225834,0.775166,0.77842031,0.53063093,0.24617932,0.13127478,0.10381947,0.09773357,0.10650625,0.10084718,0.132796,0.20757977,0.38015322,0.51071339,0.51036177,0.37886283,0.21233137,0.12898881,0.10138557,0.10090894,0.10336648,0.10138996,0.09517959,0.09790232,0.09702077,0.09026892,0.09550434,0.0948996,0.1032167,0.10369791,0.1091534,0.08802587,0.11070216,0.09511915,0.09434977,0.11289584,0.10234497,0.09680956,0.09095352,0.09636354,
CAL-09,0,sim_Raman785_demo,1,mixture,calibration,2.5,10.0,87.5,0.06413382,0.04853096,0.06405643,0.05371409,0.05826755,0.06441541,0.0586505,0.05234174,0.05974856,0.04483149,0.05110729,0.08301378,0.16193172,0.15563494,0.08704844,0.05732065,0.05754236,0.05660905,0.11429408,0.1450184,0.10475899,0.08014586,0.20160035,0.467192,0.63852563,0.47607489,0.20149709,0.08035584,0.0564662,0.08074229,0.20829205,0.45127331,0.73322905,0.7531919,0.48422414,0.21096026,0.09838999,0.05335006,0.05507809,0.04775447,0.06324282,0.08850337,0.17906789,0.30941309,0.4532671,0.4470344,0.32505173,0.16540889,0.0792102,0.06788948,0.05972875,0.06087656,0.05013054,0.05639045,0.05307545,0.0675321,0.05513123,0.0576104,0.0535568,0.04904173,0.06084569,0.06551196,0.05150733,0.05566261,0.05559255,0.06100287,0.05025988,0.05378366,0.06320598,0.05014034,0.06112492,
CAL-09,0,sim_Raman785_demo,2,mixture,calibration,2.5,10.0,87.5,0.07469827,0.08029342,0.07712041,0.07594002,0.07703393,0.08828163,0.07647418,0.09482623,0.07968894,0.08451252,0.0850042,0.11358806,0.18224664,0.17050445,0.10677918,0.08369573,0.07427628,0.07491896,0.12986328,0.15882675,0.11686903,0.10343488,0.23688051,0.4992329,0.64410245,0.49442455,0.23516565,0.10937362,0.09042029,0.11102567,0.23721104,0.47066524,0.75169324,0.77204677,0.50549662,0.2341508,0.11331856,0.0792899,0.08179203,0.08788833,0.08836702,0.10870347,0.17296168,0.34007832,0.48990556,0.47626473,0.32943963,0.18916703,0.10441899,0.08766566,0.07363439,0.08407939,0.08440569,0.07569848,0.08162985,0.07504316,0.07583537,0.07990298,0.0815438,0.07388478,0.07504446,0.08192266,0.08248,0.07856879,0.07473395,0.08935827,0.0828385,0.0669057,0.07029879,0.07591721,0.07788956,
CAL-10,0,sim_Raman785_demo,1,mixture,calibration,5.0,0.0,95.0,0.05591717,0.06658223,0.06361039,0.05688993,0.06585155,0.05228789,0.06892739,0.06795436,0.07131499,0.07270405,0.0628658,0.06010416,0.06284976,0.06432671,0.0678776,0.06566165,0.06116041,0.06231275,0.06950866,0.07449293,0.07252143,0.09256073,0.22602083,0.50046971,0.66658675,0.508893,0.21855587,0.08669749,0.0679122,0.08288942,0.24811727,0.48932248,0.77748548,0.7761363,0.48195824,0.18542514,0.09280666,0.05684389,0.06513621,0.07307454,0.07305333,0.09213499,0.18534008,0.34847546,0.49373428,0.49232639,0.35290356,0.19010699,0.10543939,0.06835442,0.06931504,0.07674999,0.06097584,0.06209241,0.06384523,0.0612457,0.05810582,0.05453891,0.05499109,0.05917052,0.06991511,0.07740111,0.06652428,0.0646486,0.06469719,0.05582093,0.06139442,0.06530393,0.05818642,0.06206752,0.05660808,
CAL-10,0,sim_Raman785_demo,2,mixture,calibration,5.0,0.0,95.0,0.10373191,0.0917242,0.08806475,0.10397826,0.09615895,0.08962206,0.10108979,0.10053153,0.10303439,0.08765611,0.10669207,0.10218127,0.10297658,0.1031673,0.09687838,0.09381358,0.09697004,0.09519205,0.09949084,0.10147463,0.10277059,0.12137824,0.27088028,0.55159945,0.70782882,0.55619355,0.26378756,0.13312824,0.1026517,0.12173803,0.27976876,0.52970734,0.81784333,0.81150408,0.52905816,0.23089306,0.12898304,0.10155591,0.0862621,0.09884798,0.11388977,0.12840209,0.21805592,0.38790514,0.52382264,0.52326715,0.38121635,0.22761676,0.14025979,0.10475421,0.11247404,0.10036916,0.09706821,0.10080673,0.09823017,0.10402042,0.10150519,0.10093433,0.08586373,0.10205672,0.10692082,0.09382968,0.09867362,0.09199507,0.09698998,0.10670592,0.09401964,0.1055579,0.10109829,0.09954358,0.09699553,
CAL-11,0,sim_Raman785_demo,1,mixture,calibration,5.0,2.5,92.5,0.12038151,0.12230164,0.1217742,0.12237241,0.11732673,0.12903182,0.1210915,0.12956849,0.12441016,0.11964731,0.12613946,0.14284947,0.15569152,0.15381435,0.13243729,0.12241505,0.12688052,0.12277184,0.13923305,0.15993908,0.1442935,0.1489194,0.29026619,0.55649275,0.71981278,0.5603676,0.2906702,0.15343988,0.13237511,0.16466012,0.29800769,0.54535636,0.81424109,0.8235691,0.54311922,0.26428358,0.15129659,0.12633353,0.11603834,0.11856271,0.13286646,0.15578255,0.24369435,0.40215803,0.54805661,0.55455627,0.40443446,0.25122001,0.16443497,0.12386648,0.11228444,0.11374124,0.1316369,0.12543277,0.12721551,0.11365947,0.12193497,0.12050009,0.12995202,0.12254069,0.14889584,0.14421522,0.12901642,0.13729403,0.13574218,0.12486999,0.13192598,0.12390009,0.13043182,0.13382563,0.12421358,
CAL-11,0,sim_Raman785_demo,2,mixture,calibration,5.0,2.5,92.5,0.05554725,0.06253283,0.06637692,0.06339468,0.06455363,0.07233526,0.06106219,0.0660938,0.0617506,0.06996409,0.05782796,0.06906424,0.08026372,0.0835547,0.06306272,0.07032046,0.06437871,0.06453958,0.07082365,0.08920966,0.08241625,0.09738719,0.22221815,0.49564362,0.66776828,0.50502506,0.23428677,0.09741912,0.07295715,0.08814411,0.23675296,0.48634193,0.76705087,0.75439489,0.4769685,0.21295104,0.09460002,0.07312206,0.0621488,0.06473819,0.07060529,0.09734642,0.18771089,0.33328156,0.4908765,0.49075812,0.35209629,0.18059627,0.10516089,0.07559087,0.06145784,0.07277313,0.06687452,0.05615408,0.06449592,0.06343905,0.05217439,0.05949791,0.06923427,0.05775511,0.07891735,0.07191647,0.06766141,0.06321751,0.06400205,0.07224988,0.06602535,0.06444087,0.06555188,0.07296127,0.06454784,
CAL-12,0,sim_Raman785_demo,1,mixture,calibration,5.0,5.0,90.0,0.07933598,0.07942349,0.07525135,0.07816639,0.08247541,0.08836809,0.07754857,0.0749814,0.08324313,0.07564871,0.06841015,0.08924367,0.11860557,0.13139535,0.08910498,0.07693577,0.07206957,0.07533266,0.0909743,0.12798655,0.10166545,0.09614999,0.23169949,0.49752049,0.6565935,0.50380967,0.22314022,0.10153866,0.07855268,0.09977645,0.23996916,0.48471758,0.76071852,0.76375983,0.49279179,0.21244425,0.10045228,0.0763518,0.07429685,0.074115,0.08296097,0.1013834,0.1955618,0.34407365,0.48188517,0.49003192,0.34478057,0.18663364,0.11206583,0.0792227,0.07406436,0.07877853,0.08317812,0.07022628,0.06747711,0.08363595,0.08289437,0.07693366,0.07046858,0.08272266,0.08949328,0.08896825,0.07554845,0.07323445,0.07517285,0.07929028,0.08645536,0.07393397,0.07238252,0.06909214,0.08422287,
CAL-12,0,sim_Raman785_demo,2,mixture,calibration,5.0,5.0,90.0,0.120907,0.11157396,0.11366469,0.11927669,0.11437794,0.11721518,0.10342687,0.11989565,0.11547149,0.12397858,0.10640736,0.12853294,0.16219321,0.16766847,0.13892141,0.10760061,0.11068917,0.12158087,0.14547792,0.16305434,0.13259635,0.15103074,0.28171352,0.53812588,0.68864673,0.54274798,0.27242304,0.14841075,0.12215456,0.13422649,0.29034819,0.53867047,0.79647346,0.81008939,0.53830391,0.25219646,0.13776236,0.10763482,0.11250314,0.11181308,0.12194991,0.14955505,0.23023363,0.37563673,0.53815387,0.53856406,0.38920535,0.2440902,0.14824936,0.12173194,0.11444026,0.11894042,0.11006444,0.10786318,0.11282511,0.12220093,0.11535625,0.11400983,0.10794075,0.11761394,0.12335174,0.11071201,0.12126397,0.11158088,0.11841608,0.11844568,0.1162812,0.11361075,0.10675515,0.1258195,0.12194532,
CAL-13,0,sim_Raman785_demo,1,mixture,calibration,5.0,7.5,87.5,0.10845551,0.11157965,0.09981651,0.1093349,0.10613261,0.10221642,0.10439174,0.10865658,0.09862871,0.09542507,0.10584663,0.12832681,0.17958374,0.19260398,0.12179315,0.10554604,0.11959784,0.11532733,0.13434965,0.18572433,0.13944264,0.14310587,0.25700629,0.52422106,0.67833845,0.52426193,0.25679632,0.13693888,0.10647334,0.14032354,0.27301417,0.5089874,0.7817712,0.78052586,0.51520189,0.26303614,0.13641022,0.09833086,0.10478217,0.10211773,0.11651548,0.13207085,0.21492151,0.3761096,0.51329818,0.51234796,0.38191304,0.21881612,0.14761254,0.10826746,0.10113867,0.11433565,0.10743631,0.10327202,0.09784018,0.10552844,0.10357833,0.1227686,0.11301754,0.11098786,0.12373465,0.12836427,0.10865901,0.10681073,0.11206592,0.1063232,0.11064283,0.10110581,0.10536363,0.09989377,0.1162083,
CAL-13,0,sim_Raman785_demo,2,mixture,calibration,5.0,7.5,87.5,0.09855918,0.09172893,0.07894305,0.08215845,0.09861576,0.08880293,0.09480545,0.09119802,0.0952793,0.09546068,0.08946099,0.11941748,0.16693066,0.17336601,0.11116002,0.08992969,0.09331797,0.10038391,0.13579184,0.17419425,0.12783739,0.12380251,0.25249382,0.51861837,0.66556799,0.51284043,0.24188708,0.1240384,0.10046248,0.10944966,0.26453546,0.49395471,0.76787929,0.77782113,0.5180075,0.23250045,0.11903986,0.09568538,0.09207537,0.09803416,0.08738514,0.12312761,0.20613917,0.36680711,0.48909888,0.49600055,0.3547404,0.20935731,0.12130603,0.09784207,0.10016891,0.09532565,0.10352775,0.09620625,0.10096477,0.09643881,0.09522552,0.09469193,0.08662019,0.09578506,0.094293,0.10102215,0.09242414,0.09391072,0.08793185,0.09463966,0.09350473,0.09785634,0.09148999,0.08974367,0.08946318,
CAL-14,0,sim_Raman785_demo,1,mixture,calibration,5.0,10.0,85.0,0.03906334,0.03178188,0.02983626,0.03063709,0.0345156,0.0268607,0.03198009,0.03201911,0.03657358,0.03165589,0.03295409,0.06635885,0.12258296,0.13225377,0.06335702,0.02867733,0.02357107,0.02295773,0.08840213,0.12541543,0.0770586,0.06393411,0.17799186,0.43478802,0.58088965,0.44027929,0.18575636,0.06162225,0.0365919,0.05794597,0.20926893,0.41867918,0.68758369,0.70263023,0.45645856,0.18608223,0.06262882,0.02855828,0.02901543,0.03627067,0.03909149,0.06644481,0.13760667,0.28869432,0.42565256,0.41672794,0.29416116,0.1476318,0.07291335,0.04240788,0.03720255,0.02353424,0.0233246,0.02754407,0.03717051,0.04042591,0.04229939,0.03377344,0.01863535,0.03977258,0.04166058,0.03684897,0.03294199,0.02250496,0.02178905,0.02910957,0.0374229,0.01969553,0.02663814,0.03851258,0.02851551,
CAL-14,0,sim_Raman785_demo,2,mixture,calibration,5.0,10.0,85.0,0.04640284,0.05302007,0.05923648,0.05775749,0.05019682,0.04781931,0.05652883,0.06097864,0.05196201,0.05544853,0.05308131,0.08661481,0.14172173,0.15487782,0.08679555,0.05620292,0.03971081,0.06277728,0.10450453,0.14018297,0.10755741,0.08238817,0.20793425,0.45097937,0.6154459,0.45972422,0.20083757,0.07180192,0.05285516,0.06950481,0.22570096,0.43858217,0.70795121,0.72475224,0.46099047,0.1995725,0.08566713,0.04342192,0.04788496,0.05961913,0.05278464,0.07591912,0.15197457,0.31051218,0.44181089,0.4404767,0.31057686,0.16442852,0.08948442,0.05945979,0.0611665,0.05312958,0.04414531,0.06298355,0.04015323,0.05027587,0.05402661,0.04657759,0.04821502,0.05060955,0.06780652,0.06584098,0.05418329,0.04454686,0.05239561,0.05652725,0.05525463,0.04127673,0.05280758,0.05242194,0.05348292,
CAL-15,0,sim_Raman785_demo,1,mixture,calibration,7.5,0.0,92.5,0.08669355,0.08017578,0.07957689,0.08449211,0.0730813,0.08348136,0.08835706,0.07941919,0.07955178,0.07579085,0.09019328,0.08892393,0.08240065,0.08009699,0.0868698,0.08664569,0.08930794,0.08071983,0.08556154,0.09329831,0.09390963,0.11348356,0.251196,0.52888126,0.68554954,0.52502577,0.24275348,0.1217545,0.09840517,0.10351301,0.29499122,0.5095456,0.78233397,0.78571846,0.49907559,0.22178801,0.10836128,0.0845363,0.08778515,0.08085433,0.08780282,0.12060253,0.20001224,0.35834365,0.50032303,0.50184273,0.3764639,0.20906279,0.11340288,0.0920893,0.10450157,0.09368589,0.08350305,0.082599,0.08508436,0.08692266,0.09591424,0.08961049,0.08031504,0.10005698,0.11016602,0.10357011,0.09489673,0.08526991,0.08009058,0.0804702,0.08194536,0.08775048,0.08950876,0.08735383,0.08601764,
CAL-15,0,sim_Raman785_demo,2,mixture,calibration,7.5,0.0,92.5,0.06951641,0.07219612,0.06158551,0.07048798,0.06404213,0.07022099,0.08172812,0.06297645,0.07473183,0.07711333,0.07255607,0.07585672,0.07325348,0.06539359,0.0717848,0.07657898,0.06749631,0.07074866,0.08004658,0.08228182,0.09374402,0.10648467,0.2364546,0.51058838,0.67450672,0.50885245,0.23564316,0.10888564,0.07637389,0.10480744,0.27393698,0.49070695,0.76654175,0.76672207,0.48512179,0.20709903,0.09080868,0.07662231,0.07351795,0.06984174,0.07626559,0.09870672,0.18309007,0.35904278,0.49037721,0.49457163,0.35245094,0.19609352,0.11288464,0.07542599,0.07292286,0.07508316,0.06947881,0.05623626,0.0736622,0.07214426,0.0705707,0.0745288,0.0580241,0.06871264,0.07994288,0.08837182,0.07945153,0.08035938,0.07639564,0.07717108,0.07417798,0.07452506,0.07724376,0.07734718,0.07395682,
CAL-16,0,sim_Raman785_demo,1,mixture,calibration,7.5,2.5,90.0,0.08026389,0.07858009,0.07817777,0.08650693,0.07862561,0.07179252,0.0870824,0.08358286,0.06920058,0.07990045,0.08423894,0.0796384,0.09576743,0.10443871,0.09373065,0.081422,0.08507943,0.082333,0.09125294,0.10976938,0.09432871,0.09920428,0.24607712,0.51546958,0.65765982,0.51633671,0.23723386,0.10857828,0.07703284,0.11511931,0.28164518,0.48823561,0.76306889,0.76326023,0.49320674,0.23014007,0.10128056,0.08661701,0.08040226,0.0856276,0.09439872,0.10770774,0.20258075,0.3549639,0.49487372,0.49560506,0.35200683,0.19561354,0.11371428,0.09694775,0.07499431,0.08430145,0.07961675,0.07492718,0.06733756,0.08447508,0.07368396,0.07917582,0.081076,0.08533022,0.09579774,0.09822381,0.09055653,0.0716779,0.07912764,0.08182314,0.0736993,0.08233626,0.07578489,0.07980318,0.07924802,
CAL-16,0,sim_Raman785_demo,2,mixture,calibration,7.5,2.5,90.0,0.10878516,0.08978054,0.09363382,0.10583645,0.1044039,0.10314259,0.10526263,0.10407367,0.10131894,0.11066677,0.11335087,0.09989948,0.12362039,0.12778646,0.0993668,0.10296563,0.10426753,0.10294915,0.1277069,0.13613354,0.12724741,0.1349742,0.26507291,0.53855585,0.67998498,0.53890784,0.25750685,0.14244335,0.09990489,0.13486287,0.30540036,0.5159226,0.79267522,0.79221688,0.50924571,0.24774912,0.12887152,0.10217225,0.09924049,0.10511084,0.10335927,0.14079886,0.21436707,0.37669379,0.51562832,0.51969544,0.37519087,0.21298598,0.13450125,0.10010859,0.09992403,0.09320839,0.10856839,0.11241339,0.09346278,0.09811186,0.10338979,0.10504538,0.10806756,0.09843373,0.13000969,0.12290628,0.10057778,0.10301882,0.11076516,0.08507935,0.0969226,0.10563241,0.10876466,0.10523416,0.10493913,
CAL-17,0,sim_Raman785_demo,1,mixture,calibration,7.5,5.0,87.5,0.09817036,0.11160935,0.11583883,0.11089295,0.11323378,0.11541766,0.10886052,0.11501709,0.10956988,0.10905739,0.12281446,0.1226553,0.14973661,0.15153806,0.12269758,0.11513202,0.0986513,0.10987972,0.13689084,0.16834076,0.14403852,0.14312946,0.26285597,0.53222198,0.68826199,0.53621815,0.25790233,0.12792273,0.11128938,0.13698736,0.30743531,0.5097202,0.79506661,0.79130127,0.52346462,0.2513136,0.12604018,0.10458071,0.10460894,0.12508965,0.11806236,0.14644752,0.22753966,0.37792675,0.52306277,0.50935496,0.37792627,0.23235348,0.14820856,0.11136425,0.11464623,0.11907729,0.09878146,0.12112882,0.11509918,0.10355728,0.12538437,0.10407864,0.12012615,0.10912408,0.12837336,0.13484199,0.12465685,0.10343731,0.11553979,0.11616558,0.11162652,0.11758834,0.11275659,0.11577622,0.12167279,
CAL-17,0,sim_Raman785_demo,2,mixture,calibration,7.5,5.0,87.5,0.08810372,0.08978655,0.09540987,0.09332825,0.08820938,0.10262009,0.09036974,0.09253916,0.09789551,0.08485609,0.09504445,0.1002078,0.14127568,0.14065059,0.094705,0.08696206,0.08135722,0.09826864,0.11656447,0.1459007,0.12256602,0.11356119,0.24204354,0.51252479,0.66154288,0.51663658,0.24655897,0.11899814,0.09348038,0.11016635,0.29022305,0.4924425,0.77002593,0.76815767,0.50382307,0.23277626,0.10436881,0.08381374,0.09277278,0.09201607,0.09294966,0.11228201,0.20499216,0.35369579,0.48934593,0.49408622,0.35605046,0.21316558,0.13011292,0.09853465,0.09349952,0.08815619,0.09329897,0.09537104,0.09208741,0.07710042,0.10415482,0.0858427,0.08127211,0.10143431,0.11677407,0.1087936,0.08382037,0.0917305,0.10030586,0.09714555,0.09317488,0.09102581,0.08998517,0.09020196,0.08463843,
CAL-18,0,sim_Raman785_demo,1,mixture,calibration,7.5,7.5,85.0,0.09202326,0.09029334,0.10435116,0.10057121,0.10198039,0.10128916,0.10626976,0.09369228,0.08662244,0.09690959,0.09240922,0.11936449,0.17510413,0.18005468,0.11223988,0.10152723,0.10081027,0.10757069,0.12063372,0.17277843,0.1263938,0.13895039,0.25770551,0.50060081,0.65892461,0.49807348,0.25408433,0.13870381,0.10377168,0.11825177,0.28482455,0.48388946,0.75541233,0.76818339,0.51867211,0.23518889,0.1287677,0.0928462,0.10482383,0.08879958,0.11125018,0.13933719,0.20081735,0.36835812,0.48829758,0.49564965,0.36772565,0.22168181,0.1309145,0.10680251,0.09225222,0.1087696,0.10374236,0.10958526,0.10665054,0.09980275,0.09796717,0.10610566,0.09143382,0.11320941,0.12055917,0.11850184,0.10279769,0.10188632,0.09817217,0.09534437,0.09267437,0.09478018,0.10295773,0.09306476,0.09609208,
CAL-18,0,sim_Raman785_demo,2,mixture,calibration,7.5,7.5,85.0,0.09713208,0.10155569,0.09906702,0.10265592,0.0956458,0.09387357,0.09274165,0.09441756,0.09245866,0.10037743,0.10045879,0.12245905,0.17684016,0.1697188,0.12241885,0.11301599,0.10755606,0.10877469,0.13245993,0.17280789,0.13411901,0.12727225,0.24773488,0.50505481,0.66181548,0.50016479,0.24951604,0.12040682,0.09491586,0.13644378,0.29365519,0.49339875,0.75975061,0.77852621,0.52186877,0.24110536,0.12492951,0.11558611,0.10404394,0.09658684,0.10956431,0.12038319,0.20114007,0.35213311,0.48589311,0.48682208,0.3635842,0.2291859,0.12597887,0.11922814,0.10017768,0.0966697,0.10588753,0.10324085,0.09572028,0.10528892,0.10106205,0.09773957,0.10083146,0.11487354,0.12368257,0.11743303,0.09373898,0.10424183,0.08619823,0.11187509,0.08975104,0.114383,0.10199452,0.10772999,0.09309839,
CAL-19,0,sim_Raman785_demo,1,mixture,calibration,7.5,10.0,82.5,0.0419595,0.03733779,0.03940219,0.04535719,0.04429327,0.03806984,0.04649248,0.05211193,0.05065486,0.04551334,0.04353202,0.08002078,0.1303314,0.14166883,0.0749672,0.04078001,0.0575588,0.05384939,0.08683508,0.14322885,0.10348857,0.06190463,0.1814738,0.42973447,0.58150625,0.43993345,0.19116878,0.07098404,0.05359836,0.07592193,0.22973451,0.4242948,0.67628573,0.69237148,0.44026653,0.19737329,0.06877337,0.05002728,0.05328558,0.05229504,0.06030488,0.07890995,0.14242112,0.28343671,0.43069831,0.43400955,0.30406476,0.16523505,0.07177636,0.04683232,0.04698987,0.03709032,0.04195029,0.03557279,0.04626329,0.04775355,0.04357059,0.04975835,0.04374756,0.04994087,0.07232571,0.0673914,0.05405856,0.04326057,0.05474782,0.0406218,0.04238525,0.04093551,0.04698814,0.04795927,0.04600979,
CAL-19,0,sim_Raman785_demo,2,mixture,calibration,7.5,10.0,82.5,0.05396507,0.04902391,0.06376042,0.05326739,0.05930133,0.06011192,0.04259258,0.05051427,0.05381351,0.05449054,0.05958133,0.09781718,0.15591463,0.16750207,0.09535735,0.06174018,0.05135663,0.05957434,0.10909018,0.145551,0.10105562,0.08102832,0.20916735,0.46051923,0.60146728,0.45899274,0.20092085,0.08079479,0.06739084,0.09005956,0.24433008,0.44541411,0.70690092,0.70865921,0.46794155,0.20213032,0.09288034,0.05843957,0.0621598,0.05434412,0.07483252,0.09939022,0.16760232,0.31408817,0.44432082,0.451848,0.30710493,0.18479168,0.08914329,0.06377633,0.06384768,0.05592719,0.06577305,0.06160509,0.05974612,0.06072092,0.07153173,0.05695759,0.04739703,0.06274821,0.07826837,0.07833312,0.05872083,0.06602625,0.05515415,0.06427966,0.05289698,0.06032247,0.06169869,0.06068587,0.05614415,
CAL-20,0,sim_Raman785_demo,1,mixture,calibration,10.0,0.0,90.0,0.14053522,0.12939483,0.13098682,0.15343507,0.14700742,0.13148461,0.13142697,0.14005058,0.13033147,0.14100284,0.13281764,0.13417597,0.13291735,0.12771487,0.14049942,0.13186563,0.14932657,0.1374235,0.13194756,0.15797242,0.15140721,0.16228621,0.29400733,0.5715643,0.7177301,0.57329171,0.2931708,0.1547646,0.12835099,0.1632744,0.34944714,0.55975117,0.81664537,0.8191626,0.53521755,0.25396262,0.14653122,0.1351241,0.14171879,0.13957578,0.13444242,0.15991443,0.25033567,0.4007761,0.56040536,0.53706397,0.40904663,0.26458044,0.17474669,0.14122407,0.13041885,0.12992735,0.13766196,0.13578833,0.13884057,0.12762471,0.13732607,0.12376872,0.14248446,0.14243332,0.16685756,0.16309889,0.14341962,0.1365731,0.13232163,0.12971804,0.1304654,0.13723311,0.13978826,0.12913729,0.13216328,
CAL-20,0,sim_Raman785_demo,2,mixture,calibration,10.0,0.0,90.0,0.10897304,0.10615972,0.11067216,0.10732861,0.10791808,0.12334618,0.11129285,0.11032682,0.1079242,0.11971581,0.11315109,0.10908631,0.11350034,0.1148001,0.11569418,0.12034432,0.10538588,0.11844202,0.12000947,0.13551202,0.12507111,0.14932228,0.277192,0.5382467,0.69831911,0.54726443,0.27341865,0.13929697,0.12632684,0.13951561,0.3389385,0.52554459,0.80736645,0.81164705,0.50528153,0.25200174,0.12681106,0.12279945,0.11537709,0.11312004,0.12546119,0.14466758,0.22610851,0.38778907,0.52827269,0.52585181,0.39479359,0.24769124,0.15498779,0.12309622,0.11441574,0.1197156,0.11341642,0.11707089,0.11871466,0.11358422,0.11748114,0.12515614,0.11170766,0.12781908,0.14346724,0.14888599,0.11696102,0.11548873,0.10475669,0.10662299,0.11882874,0.12397029,0.1083941,0.1145376,0.12382861,
CAL-21,0,sim_Raman785_demo,1,mixture,calibration,10.0,2.5,87.5,0.0655622,0.06954906,0.06972215,0.06395728,0.07029017,0.06290375,0.0749906,0.0680148,0.07165932,0.07309225,0.07463778,0.06901443,0.09637768,0.09401166,0.07051963,0.07205065,0.06580787,0.05958106,0.08669183,0.10454131,0.08105283,0.09509952,0.21623123,0.49328484,0.63752775,0.46965013,0.22874269,0.09171769,0.07354135,0.09092087,0.28243744,0.4698625,0.73447255,0.75292098,0.46874286,0.20074742,0.09223943,0.07113215,0.08063861,0.07338712,0.08004761,0.08941642,0.18879643,0.33616436,0.46374294,0.4683204,0.34712719,0.18759838,0.11028177,0.08133973,0.07296135,0.06849003,0.06958339,0.0600875,0.06758794,0.0565052,0.06803694,0.07346673,0.06999161,0.08438098,0.08659481,0.08263725,0.06993367,0.06922379,0.06210641,0.06948919,0.06446857,0.06581072,0.06593185,0.06374891,0.07862874,
CAL-21,0,sim_Raman785_demo,2,mixture,calibration,10.0,2.5,87.5,0.07095824,0.07700334,0.06651484,0.06572787,0.06871706,0.06661829,0.06570229,0.06865842,0.07543156,0.07233434,0.07643772,0.07735201,0.09034227,0.09199265,0.08520535,0.06517815,0.06819449,0.0789038,0.08475014,0.11549534,0.09490878,0.10741208,0.22669368,0.47558983,0.64112898,0.49153263,0.23157163,0.10363164,0.07816232,0.09236819,0.30061507,0.47422713,0.74895026,0.74085754,0.46592435,0.20283882,0.09878743,0.07729744,0.0642813,0.06586073,0.07250722,0.10121498,0.18475195,0.33343354,0.46455956,0.47279939,0.33260426,0.19271482,0.11101074,0.08019955,0.07080076,0.07252047,0.06340669,0.05967658,0.06957674,0.07599094,0.06072433,0.05943254,0.06846955,0.08420187,0.10202383,0.09589663,0.07379597,0.06437298,0.07589629,0.08129596,0.07280636,0.06316971,0.0628944,0.07279933,0.07295037,
CAL-22,0,sim_Raman785_demo,1,mixture,calibration,10.0,5.0,85.0,0.06517814,0.06065865,0.06147368,0.0679025,0.07413341,0.07773986,0.07996477,0.07497158,0.06247742,0.07045774,0.07680502,0.08972898,0.1220612,0.13194719,0.08677414,0.07495843,0.0695698,0.08303911,0.08790611,0.1237657,0.10311542,0.10718538,0.22413965,0.47924483,0.63731132,0.48039115,0.22172776,0.097277,0.0820788,0.08673773,0.28662214,0.48465271,0.71705061,0.72778388,0.47540868,0.21793052,0.09450073,0.08820261,0.06686451,0.07945633,0.07372815,0.1033516,0.1746164,0.31885488,0.46642739,0.45548155,0.33078339,0.19223713,0.11591651,0.09151289,0.07408094,0.08364433,0.06689603,0.07362654,0.07922401,0.06686952,0.06568098,0.07120885,0.07534495,0.07568267,0.1040266,0.1067963,0.08326663,0.06924277,0.07757605,0.07985131,0.06814408,0.06772894,0.07513768,0.06802975,0.06869891,
CAL-22,0,sim_Raman785_demo,2,mixture,calibration,10.0,5.0,85.0,0.09176233,0.08618743,0.09023798,0.09362935,0.09141532,0.09774004,0.08134824,0.0964643,0.08989877,0.08656409,0.09170508,0.1036837,0.13711626,0.14300705,0.11016869,0.09552539,0.09710593,0.08221964,0.11539708,0.14999872,0.13105783,0.11890789,0.24596198,0.49849192,0.64288257,0.50459579,0.24344983,0.11760655,0.0840832,0.12430177,0.30556594,0.49200229,0.74877328,0.75555064,0.50472705,0.23517467,0.11031924,0.10142912,0.08394922,0.09395845,0.09962627,0.12436055,0.2130606,0.35209666,0.47926158,0.49071093,0.35894375,0.22036309,0.12947255,0.10091401,0.09407113,0.08930361,0.08933619,0.09207013,0.10241675,0.08418017,0.09667706,0.09124999,0.08731301,0.08574657,0.11942866,0.11772701,0.09984273,0.10220069,0.09105131,0.09573846,0.09630128,0.09030753,0.10555906,0.0963264,0.0879189,
CAL-23,0,sim_Raman785_demo,1,mixture,calibration,10.0,7.5,82.5,0.12565596,0.13651104,0.11953527,0.12682932,0.13104202,0.1322513,0.14068015,0.12922527,0.12632402,0.12498241,0.12655024,0.14943908,0.20143505,0.19188501,0.15231822,0.13176437,0.11752281,0.13142708,0.16289971,0.20893212,0.18333609,0.15757083,0.26854758,0.5168958,0.6729774,0.52445842,0.26832023,0.15112703,0.13391968,0.14819924,0.33916797,0.50700415,0.76848509,0.77291501,0.52365508,0.26267382,0.14751138,0.13100189,0.13523129,0.13047158,0.14082934,0.14289526,0.2401196,0.37957924,0.50755104,0.51409441,0.38511962,0.23794336,0.1580038,0.13549364,0.1188594,0.13147663,0.12304758,0.12679778,0.13950056,0.1367623,0.13253896,0.1280046,0.11960273,0.13412947,0.1499203,0.14802852,0.13482094,0.12964451,0.13626653,0.1254356,0.1340324,0.11627888,0.13607604,0.13266281,0.13382441,
CAL-23,0,sim_Raman785_demo,2,mixture,calibration,10.0,7.5,82.5,0.0814158,0.08169965,0.07170627,0.08342976,0.07915522,0.07753725,0.07865891,0.07429704,0.08161053,0.06185767,0.0716435,0.1004073,0.15048709,0.14899099,0.10149773,0.08517863,0.07812488,0.07820326,0.10315171,0.15801511,0.11680273,0.10596929,0.2263747,0.47368364,0.62133018,0.47705816,0.22173131,0.09763485,0.07504343,0.0973128,0.27719047,0.46017209,0.72276377,0.72892513,0.477327,0.2167437,0.10741553,0.09620861,0.07686011,0.07407318,0.08324706,0.10415826,0.17610252,0.32871665,0.46229689,0.46680837,0.33795057,0.20456791,0.11187364,0.07714709,0.09338493,0.07205078,0.08092526,0.07324496,0.07955636,0.08400562,0.07632335,0.06644046,0.07588399,0.07590441,0.09566582,0.10562678,0.0778671,0.07633877,0.07824687,0.08753161,0.07911129,0.07923283,0.07992481,0.07610559,0.082936,
CAL-24,0,sim_Raman785_demo,1,mixture,calibration,10.0,10.0,80.0,0.08902523,0.07834021,0.07581508,0.08145575,0.0658369,0.06784983,0.07845581,0.0866048,0.07345442,0.07988465,0.07721197,0.10621891,0.17676991,0.17711457,0.11010107,0.08271932,0.08293743,0.07942637,0.11314551,0.17729652,0.12544645,0.11242711,0.2188757,0.45570732,0.60188491,0.46342076,0.21785723,0.09885865,0.08185383,0.10055424,0.28777967,0.4533369,0.70493864,0.71479646,0.47540293,0.22228033,0.10731556,0.08442769,0.07201178,0.07974793,0.08760063,0.11101319,0.18523545,0.30437885,0.44433142,0.45459752,0.33869766,0.19685274,0.11578264,0.08893523,0.07702378,0.08176536,0.08001564,0.08078476,0.06495989,0.07787034,0.06690831,0.08314113,0.07893512,0.07898304,0.10260876,0.10563211,0.0783646,0.08336557,0.07635788,0.08161909,0.06999663,0.07429409,0.08710557,0.08834288,0.08636326,
CAL-24,0,sim_Raman785_demo,2,mixture,calibration,10.0,10.0,80.0,0.12201349,0.11711302,0.11318713,0.10785813,0.1169013,0.11235277,0.11689275,0.10776409,0.11901453,0.11795514,0.11425319,0.14067118,0.2066636,0.20792204,0.14993765,0.11345166,0.11327345,0.11148564,0.1547318,0.21447511,0.1600261,0.14263641,0.25501827,0.50361296,0.64514108,0.50447975,0.25666001,0.14494616,0.11490083,0.13174596,0.30990697,0.47979193,0.74241337,0.75449775,0.50414078,0.25354548,0.14064423,0.1193523,0.11108493,0.10100575,0.11176589,0.13828369,0.21215616,0.3567567,0.48814945,0.48636361,0.36949675,0.2298746,0.14495291,0.12203619,0.11032373,0.12371449,0.11252724,0.11066814,0.10909055,0.11147866,0.11829669,0.1093912,0.11598093,0.11878164,0.13315267,0.13601547,0.12793463,0.09543364,0.10691708,0.12106456,0.12227069,0.10792235,0.10277048,0.10798124,0.10255483,
BLIND-00,0,sim_Raman785_demo,1,mixture,blind_test,9.01,5.87,85.12,0.12710438,0.12235471,0.12560843,0.12216873,0.12962105,0.14182426,0.12534386,0.12561379,0.13487019,0.13124594,0.12666981,0.13801522,0.17609305,0.18377513,0.13300816,0.11759915,0.12556421,0.12051413,0.15421492,0.2010453,0.15944115,0.16693676,0.2797266,0.52783032,0.68238482,0.52887227,0.26751273,0.16169977,0.1302744,0.15197831,0.34176982,0.53171872,0.78184075,0.80545778,0.53075717,0.27240768,0.14666175,0.12844878,0.12022365,0.12725617,0.13546496,0.15482683,0.23676836,0.37992433,0.51754797,0.51598493,0.38448034,0.24776771,0.15973525,0.1191983,0.11845007,0.12524325,0.12329649,0.13365426,0.13133107,0.12803155,0.12548775,0.12116055,0.13171876,0.13074738,0.14855648,0.15942113,0.12570128,0.12865881,0.12914883,0.13742444,0.12688783,0.11752621,0.1313894,0.11580486,0.12756983,
BLIND-00,0,sim_Raman785_demo,2,mixture,blind_test,9.01,5.87,85.12,0.06695502,0.0816668,0.0638104,0.07371179,0.06475652,0.06814621,0.08682882,0.07266182,0.0706958,0.07159452,0.07231535,0.08713126,0.12721997,0.13187902,0.092475,0.06115372,0.06672666,0.06637005,0.09805318,0.13785552,0.09919076,0.10231826,0.21234772,0.46910539,0.63059435,0.48245488,0.21883521,0.08117333,0.07830388,0.09866367,0.27734202,0.46567792,0.73282655,0.7307945,0.48657239,0.2078996,0.09677643,0.07006861,0.06022773,0.08338523,0.08487701,0.11012954,0.17857664,0.32346691,0.45671217,0.45934279,0.34636847,0.20321396,0.10116897,0.07349414,0.0695631,0.05941242,0.06388794,0.07663014,0.07461485,0.07744784,0.06807302,0.06868454,0.07869127,0.07193402,0.10026378,0.08441449,0.08256781,0.06825705,0.07855283,0.06018555,0.06999712,0.07302813,0.07471107,0.06957363,0.08320974,
BLIND-01,0,sim_Raman785_demo,1,mixture,blind_test,3.43,3.67,92.9,0.0941135,0.08921131,0.08765692,0.08481628,0.09315766,0.10322749,0.0917865,0.09256351,0.09790141,0.09774369,0.08653496,0.11063827,0.1233039,0.13224363,0.09760867,0.09611994,0.09948996,0.08895895,0.11361256,0.12743168,0.11055194,0.12485202,0.25123213,0.53399047,0.67620106,0.53224607,0.26019865,0.11500485,0.10386925,0.11024589,0.25725248,0.51235586,0.79594231,0.80242982,0.51638717,0.24350549,0.11412332,0.08931368,0.09875527,0.10196773,0.09597604,0.1269121,0.19742024,0.36637483,0.51383329,0.51211422,0.3694298,0.20577889,0.13069123,0.10742028,0.09159016,0.09936764,0.09743876,0.09413445,0.08949498,0.08436429,0.09993997,0.09415083,0.07811531,0.10315864,0.10786962,0.0931201,0.07543357,0.07798645,0.09553786,0.08422123,0.08987725,0.09504347,0.07693886,0.09393132,0.10625411,
BLIND-01,0,sim_Raman785_demo,2,mixture,blind_test,3.43,3.67,92.9,0.10462125,0.0969159,0.10512353,0.09743196,0.09767918,0.10382553,0.10541064,0.09760961,0.1023422,0.09814572,0.10529555,0.10559479,0.141499,0.13650235,0.11409634,0.10300661,0.09339368,0.1091152,0.12066921,0.13851394,0.12675098,0.13031477,0.26215853,0.54283759,0.69791446,0.54125155,0.26962342,0.12936472,0.12167534,0.12607488,0.26996708,0.5253139,0.81421728,0.80805252,0.52593704,0.2397783,0.12117603,0.1040884,0.09851345,0.0961345,0.10059942,0.13807119,0.22397163,0.38988531,0.5321365,0.53661613,0.38093419,0.22157793,0.13689978,0.1018759,0.10643151,0.10861044,0.09544679,0.09883328,0.11859858,0.10625931,0.10014465,0.09934251,0.10124617,0.10289003,0.1203608,0.11501591,0.10986589,0.10152249,0.09329885,0.0902298,0.11105618,0.10443931,0.09611837,0.10445883,0.10593965,
BLIND-02,0,sim_Raman785_demo,1,mixture,blind_test,7.99,1.2,90.81,0.06297497,0.06888194,0.06359551,0.06127111,0.0626307,0.06570458,0.07435587,0.05791766,0.06054075,0.06898528,0.06663359,0.05980956,0.06906812,0.06211107,0.06058354,0.06161357,0.06530341,0.05974519,0.07240975,0.090218,0.07732713,0.08941374,0.22141786,0.50273428,0.65387149,0.4857751,0.21685942,0.09003304,0.05484366,0.09857926,0.2566072,0.47120834,0.73732198,0.75843608,0.46958312,0.20181917,0.07778151,0.0652021,0.05824336,0.07198459,0.06554229,0.09041388,0.1776281,0.33532728,0.47535758,0.48079925,0.34524544,0.19267871,0.10536071,0.06739703,0.06286937,0.06118847,0.07004722,0.06151325,0.06112718,0.0556834,0.0625397,0.066867,0.048324,0.06597529,0.07664511,0.0823572,0.0643276,0.07151988,0.06390225,0.06472977,0.06216913,0.0471219,0.06316991,0.06436664,0.05681054,
BLIND-02,0,sim_Raman785_demo,2,mixture,blind_test,7.99,1.2,90.81,0.08616321,0.07799006,0.08841447,0.08481475,0.08615022,0.08893686,0.07808593,0.0862805,0.07397752,0.07998584,0.0881971,0.08668802,0.10401057,0.0921692,0.08568101,0.08143252,0.08101638,0.08866648,0.10379683,0.09951066,0.09788825,0.1084775,0.24336453,0.51604781,0.67729149,0.51697241,0.24553767,0.12485894,0.08407582,0.10766857,0.29063888,0.50221007,0.77504896,0.78918918,0.50059691,0.21863909,0.10912765,0.08962017,0.08283506,0.07631909,0.09437388,0.11759396,0.2029759,0.36589142,0.49963086,0.50284118,0.36414541,0.21178262,0.11849021,0.09042503,0.08343548,0.07882887,0.08754101,0.0928804,0.09451757,0.07938242,0.08283611,0.08309504,0.08087286,0.08353186,0.10294019,0.09720353,0.0847616,0.08212618,0.07013405,0.07880865,0.07543495,0.08423493,0.07881947,0.08961156,0.09060237,
BLIND-03,0,sim_Raman785_demo,1,mixture,blind_test,7.54,6.66,85.8,0.06106526,0.0623814,0.06387689,0.06611523,0.06823542,0.06969781,0.05699823,0.06911639,0.06198771,0.06974427,0.06188486,0.08866637,0.12756627,0.12630669,0.08799613,0.06185993,0.06486535,0.0671281,0.09935677,0.13422295,0.10439479,0.08586359,0.2252643,0.47897559,0.62558902,0.46083624,0.2123589,0.09502121,0.07613605,0.08702276,0.26219882,0.45867061,0.73060598,0.7395364,0.47510999,0.20261746,0.09612564,0.07070059,0.05763724,0.07458637,0.0761833,0.09822177,0.1734506,0.32603643,0.45444463,0.46753153,0.32616315,0.18729781,0.10137766,0.0824067,0.0554787,0.07634358,0.07777718,0.05847262,0.06368417,0.06983415,0.05329652,0.08091792,0.07106394,0.07072515,0.08609504,0.08042268,0.07645768,0.07580984,0.0673351,0.06579403,0.07076124,0.08285359,0.05969934,0.07244665,0.0588181,
BLIND-03,0,sim_Raman785_demo,2,mixture,blind_test,7.54,6.66,85.8,0.05778082,0.05835891,0.05283643,0.06371938,0.07152718,0.06129669,0.05618295,0.05386783,0.04948836,0.05161922,0.05458319,0.07950667,0.12743953,0.12615217,0.07594701,0.05994503,0.06243506,0.0561498,0.08837017,0.13302096,0.10986717,0.08882159,0.21334974,0.48208539,0.62312118,0.4762218,0.20958023,0.08691382,0.05773677,0.08961967,0.24610934,0.46984617,0.71717509,0.73839858,0.4716499,0.20635153,0.08510519,0.0582638,0.05835865,0.05675302,0.06361484,0.09179686,0.17428631,0.30648331,0.44551326,0.44601393,0.32349982,0.17581199,0.09060035,0.06182215,0.05826011,0.06510124,0.06544333,0.05664366,0.06527186,0.05436418,0.06809423,0.06028919,0.05773194,0.06060494,0.07986152,0.07249865,0.05800377,0.0631285,0.05800305,0.06251434,0.05781675,0.05560818,0.05847557,0.06802076,0.05918057,
BLIND-04,0,sim_Raman785_demo,1,mixture,blind_test,1.47,8.04,90.49,0.09975017,0.10141707,0.10551493,0.09768924,0.10554612,0.09814394,0.09781005,0.09839201,0.10310974,0.10149053,0.10918647,0.1243667,0.17860557,0.17672268,0.12632124,0.10786151,0.1069504,0.10511717,0.14036168,0.16697348,0.1352575,0.12281969,0.25665515,0.53209686,0.68614235,0.53134825,0.2696931,0.12913795,0.11036708,0.12948138,0.24809218,0.51428331,0.79112663,0.80655673,0.53216733,0.26372276,0.12574855,0.11505233,0.1044935,0.10391297,0.11200644,0.13992723,0.21777885,0.36953827,0.52641436,0.52077849,0.38349461,0.22440314,0.13697515,0.10949444,0.10124516,0.11047886,0.0992568,0.10730332,0.09726803,0.1016228,0.1028363,0.09906896,0.10684188,0.11065285,0.09568683,0.11143117,0.10066453,0.10387451,0.10853405,0.10565865,0.1046306,0.11049241,0.10688045,0.10617989,0.1046629,
BLIND-04,0,sim_Raman785_demo,2,mixture,blind_test,1.47,8.04,90.49,0.12372829,0.11881343,0.10695476,0.1266916,0.11106938,0.11907457,0.10204473,0.11884601,0.1144781,0.11349272,0.11144499,0.13963201,0.18573003,0.19028529,0.13515003,0.11481579,0.10957729,0.11117524,0.13876426,0.18325223,0.15533028,0.14800539,0.27438333,0.53800009,0.69419285,0.54976529,0.28224104,0.1403982,0.11608745,0.13232299,0.26288303,0.52136504,0.79911545,0.80448592,0.54873775,0.2561559,0.14306116,0.10788946,0.11352617,0.11527835,0.11159936,0.12597291,0.2227311,0.39232053,0.52399792,0.51961611,0.38611489,0.23521196,0.13821795,0.11647845,0.10800868,0.10556364,0.10287916,0.10485325,0.12488866,0.11835138,0.10858406,0.11019462,0.11022801,0.11685703,0.11578597,0.11715838,0.11301037,0.1139038,0.11807435,0.10666508,0.11268478,0.10160402,0.11419021,0.11204369,0.10498414,
BLIND-05,0,sim_Raman785_demo,1,mixture,blind_test,8.76,6.5,84.74,0.07928406,0.07169364,0.07219296,0.08360833,0.08022965,0.06970421,0.07180869,0.07426317,0.07232259,0.07939217,0.07258367,0.09621341,0.13477208,0.12895762,0.08317456,0.07890583,0.08071892,0.08195755,0.10733956,0.14083091,0.09831898,0.10641217,0.22019627,0.48258442,0.62250192,0.48000897,0.2263385,0.09719664,0.07467346,0.0882159,0.27440525,0.46179712,0.72697448,0.73368942,0.48713616,0.20871569,0.09719937,0.07018601,0.06060459,0.07483802,0.06873426,0.08932197,0.17285596,0.33509791,0.45972911,0.45267338,0.33280297,0.19620826,0.09769401,0.06606941,0.07795526,0.06688369,0.06744817,0.07629211,0.07725193,0.06780785,0.06284436,0.07746814,0.06609744,0.082998,0.10544236,0.0877921,0.08512916,0.07157551,0.07905693,0.07156152,0.07553399,0.06478553,0.06392785,0.08054261,0.07111569,
BLIND-05,0,sim_Raman785_demo,2,mixture,blind_test,8.76,6.5,84.74,0.09471987,0.08051206,0.08758523,0.08540065,0.08958064,0.08650252,0.08962278,0.07862893,0.0767303,0.08417286,0.08352156,0.09731713,0.14774298,0.15843946,0.10737672,0.09341761,0.08157653,0.09152971,0.11832204,0.16550567,0.13036555,0.11443747,0.23624034,0.50105401,0.65032306,0.48139395,0.2411328,0.11424898,0.09720935,0.11257476,0.29852083,0.48432931,0.74902639,0.75728041,0.50367662,0.22168125,0.1164009,0.09667574,0.09806193,0.0923661,0.09571604,0.11489445,0.18829853,0.34423179,0.47263351,0.47795897,0.35207416,0.20304161,0.12655699,0.08726242,0.08523456,0.09568024,0.09280829,0.09155067,0.08746915,0.0871935,0.08840629,0.08936874,0.08954115,0.09333317,0.10768142,0.10860595,0.07821435,0.08224601,0.0824777,0.08233019,0.09155447,0.09237703,0.09374412,0.0862926,0.08428336,
BLIND-06,0,sim_Raman785_demo,1,mixture,blind_test,5.79,6.32,87.89,0.09141402,0.08289371,0.09321718,0.08719933,0.09405449,0.09249101,0.08561295,0.08874916,0.08701788,0.0894748,0.08369373,0.10779091,0.14976799,0.14769117,0.11505802,0.10335807,0.08129397,0.08490698,0.11777563,0.15453557,0.11359753,0.11450084,0.24554792,0.51090951,0.66670259,0.51099077,0.24399589,0.12751445,0.10443571,0.11356838,0.26954759,0.48959331,0.76538474,0.7786787,0.50584817,0.22284741,0.11484609,0.0852673,0.081839,0.08525155,0.08876494,0.11617774,0.19872507,0.36047609,0.49608976,0.48507925,0.35821273,0.2149311,0.11324231,0.09535697,0.08987126,0.08674323,0.08824758,0.09778294,0.09493587,0.077157,0.08930329,0.07882359,0.08934551,0.10027909,0.10196244,0.09710066,0.09248511,0.0814902,0.08712851,0.0964977,0.08764819,0.08468518,0.10698981,0.08613945,0.09608182,
BLIND-06,0,sim_Raman785_demo,2,mixture,blind_test,5.79,6.32,87.89,0.0880916,0.07995062,0.07457978,0.06996572,0.0737618,0.07082714,0.07221647,0.08497487,0.07277594,0.06279847,0.07266889,0.09117327,0.13325161,0.13237777,0.08981267,0.06519363,0.07449006,0.07381309,0.09419332,0.13186812,0.1113396,0.10396017,0.23209376,0.4942377,0.6589787,0.49834636,0.23067328,0.11992451,0.0776196,0.11166023,0.24635016,0.47008417,0.75360468,0.76925842,0.49628094,0.2272074,0.10218548,0.07403338,0.07970245,0.08247168,0.07541309,0.11434357,0.18771111,0.33647371,0.48182381,0.48376913,0.34793906,0.19278309,0.1109495,0.07269878,0.06962855,0.07727962,0.06929107,0.07937433,0.07280043,0.06838441,0.06835536,0.07341346,0.07166352,0.07898821,0.09951203,0.07942031,0.06572676,0.07260519,0.07410235,0.07256891,0.07574828,0.06827174,0.0785179,0.06386953,0.07472381,
LIB-aspartame,0,sim_Raman785_demo,1,library,reference,,,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.97e-06,0.00108621,0.03863369,0.17852647,0.10718248,0.00836045,8.473e-05,1.1e-07,0.0,0.0,0.0,0.0,1.168e-05,0.0403616,1.0,0.17756917,0.00022598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.7e-07,4.256e-05,0.00140924,0.017168,0.07694165,0.12685533,0.07694165,0.017168,0.00140924,4.256e-05,4.7e-07,0.0,0.0,0.0,4e-08,2.213e-05,0.00240254,0.0546815,0.26087087,0.26087087,0.0546815,0.00240254,2.213e-05,4e-08,0.0,0.0,0.0,0.0,0.0,aspartame
LIB-sucralose,0,sim_Raman785_demo,1,library,reference,,,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1e-08,4.35e-06,0.00060674,0.02463213,0.29096046,1.0,1.0,0.29096046,0.02463545,0.00139428,0.03915483,0.40795739,0.89106083,0.40795738,0.03915048,0.00078754,3.32e-06,0.0,0.0,0.0,0.0,1.5e-07,1.384e-05,0.00057058,0.01029303,0.08125422,0.28069038,0.42431468,0.28069038,0.08125422,0.01029303,0.00057058,1.384e-05,1.5e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sucralose
LIB-erythritol,0,sim_Raman785_demo,1,library,reference,,,,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685657,0.10685932,0.10698359,0.10974823,0.13972117,0.29337093,0.63542644,0.85485228,0.6354265,0.29337451,0.13984598,0.11215332,0.13263122,0.25820997,0.60110152,1.0,1.0,0.60110149,0.25820718,0.13250423,0.10926362,0.10702603,0.10750332,0.11279194,0.14197415,0.2400811,0.43091536,0.61226629,0.61226629,0.43091536,0.2400811,0.14197415,0.11279188,0.10749974,0.10690123,0.10685853,0.10685659,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,erythritol
COMMERCIAL-01,0,sim_Raman785_demo,1,unknown,query,1.8,,,-0.0050809,0.02906,0.16611691,0.39174072,0.53699202,0.39591456,0.16958719,0.03026443,0.00485201,0.00049616,-0.00132734,0.00971891,-0.0078998,-0.0001566,0.00685987,0.00083616,0.00075823,-0.0017753,0.0024775,0.0051934,0.02085661,-0.00414764,-0.00110432,-0.00551344,0.00425604,-0.00750036,0.00863493,-0.0045905,-0.00017633,0.00205343,0.02239125,0.00689607,0.01460605,0.13315828,0.37557295,0.6935134,0.83775341,0.69461902,0.3795855,0.13560185,0.02904774,0.00154723,0.00691458,-0.00716645,-0.00103003,0.00064423,0.00013357,0.01482621,-0.0021794,-0.00141709,-0.00671313,0.0003414,-0.00685729,0.00305155,-0.0051163,-2.46e-06,0.00422957,-0.00743273,0.00252553,-0.00192353,0.01618845,0.00536557,0.00316826,-0.00248521,0.01003087,0.00071997,-0.00321227,0.00047694,0.00307361,-0.00151572,0.00630633,
"""

In [ ]:
df = load_spectra()
meta, X_all, wn = split_spectra(df)
print(f"讀入 {len(df)} 筆光譜，{wn.min():.0f}-{wn.max():.0f} cm-1，{len(wn)} 個位移點。")
print(meta["role"].value_counts())
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_spectra(fname)
#     meta, X_all, wn = split_spectra(df)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_spectra('本機路徑或URL')")

## 2. 校正集設計與正交性檢查

5×5 全因子設計：阿斯巴甜與三氯蔗糖各取 5 個水準（0, 2.5, 5, 7.5, 10%），共 25 個配方組合，赤藻糖醇補足到 100%。理論上這個設計會讓兩個目標**完全不相關**——合併資料後第一件事就是驗證這一點：如果算出來明顯不是 0，代表資料合併或配方紀錄可能有誤植。

每個配方另外重複量測 `N_REP` 次（重新裝樣再測一次），交叉驗證要用 `sample_id` 分組，避免同一配方的重複同時落在訓練與驗證兩邊。

In [ ]:
cal_mask = (meta["split"] == "calibration").to_numpy()
blind_mask = (meta["split"] == "blind_test").to_numpy()
lib_mask = (meta["role"] == "library").to_numpy()
unk_mask = (meta["role"] == "unknown").to_numpy()

cal_meta = meta[cal_mask].reset_index(drop=True)
X_cal_raw = X_all[cal_mask]
Y_asp_cal = cal_meta["aspartame_pct"].to_numpy()
Y_suc_cal = cal_meta["sucralose_pct"].to_numpy()
g_cal = cal_meta["sample_id"].to_numpy()

# corr() computed on the UNIQUE design points (not the replicated rows) --
# duplicating each point N_REP times doesn't change the correlation, but
# computing it on the unique table is clearer pedagogically.
design_table = cal_meta.drop_duplicates("sample_id")[["aspartame_pct", "sucralose_pct"]]
corr_check = float(np.corrcoef(design_table["aspartame_pct"], design_table["sucralose_pct"])[0, 1])
print(f"corr(Y_aspartame, Y_sucralose) = {corr_check:.2e}")
assert abs(corr_check) < 1e-9, "設計應該完全正交，若不是 0 要回頭檢查資料合併/配方紀錄"
print("檢查通過：設計完全正交。")

## 3. 看幾條光譜

阿斯巴甜濃度越高，~1003 cm⁻¹（苯環呼吸振動）附近訊號越明顯；三氯蔗糖濃度越高，600–850 cm⁻¹（C–Cl 區）附近訊號越明顯。

In [ ]:
show_pos = [0, 6, 12, 18, 24]  # a spread of calibration points (rep 1 of each)
show_rows = cal_meta[cal_meta["rep"] == 1].iloc[show_pos] if "rep" in cal_meta else cal_meta.iloc[show_pos]
show_idx = show_rows.index.to_numpy()
labels = [f"asp={r.aspartame_pct:.1f}%,suc={r.sucralose_pct:.1f}%" for r in show_rows.itertuples()]
ax = plot_spectra(wn, X_cal_raw[show_idx], labels=labels, xlabel="Raman shift (cm-1)")
ax.set_title("Example calibration spectra")
plt.tight_layout()
plt.show()

## 4. 前處理比較與分組交叉驗證選 LV

比較「raw（僅置中）」與「基線扣除＋面積歸一化」兩種前處理，用 **`LeaveOneGroupOut`（以 `sample_id` 分組）** 交叉驗證比較 RMSECV（以阿斯巴甜模型為例），避免同一配方的重複測量同時出現在訓練與驗證折中。

In [ ]:
def preprocess_baseline_norm(X):
    X = np.atleast_2d(X)
    baseline = (X[:, :5].mean(axis=1, keepdims=True) + X[:, -5:].mean(axis=1, keepdims=True)) / 2.0
    Xc = X - baseline
    norms = np.sum(np.abs(Xc), axis=1, keepdims=True)
    return Xc / norms


def rmsecv_grouped(X, y, groups, max_lv=8):
    logo = LeaveOneGroupOut()
    out = []
    for lv in range(1, max_lv + 1):
        preds = np.zeros_like(y, dtype=float)
        for tr, va in logo.split(X, y, groups):
            m = PLSRegression(n_components=lv).fit(X[tr], y[tr])
            preds[va] = m.predict(X[va]).ravel()
        out.append(float(np.sqrt(np.mean((preds - y) ** 2))))
    return out


MAX_LV = 8
rmsecv_raw_asp = rmsecv_grouped(X_cal_raw, Y_asp_cal, g_cal, MAX_LV)
X_cal_pre = preprocess_baseline_norm(X_cal_raw)
rmsecv_pre_asp = rmsecv_grouped(X_cal_pre, Y_asp_cal, g_cal, MAX_LV)

print("RMSECV(阿斯巴甜) raw:          ", [round(v, 4) for v in rmsecv_raw_asp])
print("RMSECV(阿斯巴甜) baseline+norm:", [round(v, 4) for v in rmsecv_pre_asp])

use_pre = min(rmsecv_pre_asp) < min(rmsecv_raw_asp)
PREP_NAME = "基線扣除+面積歸一化" if use_pre else "raw（僅置中，PLS內建）"
print("依 RMSECV 選出的前處理：", PREP_NAME)

X_cal = X_cal_pre if use_pre else X_cal_raw
rmsecv_asp = rmsecv_pre_asp if use_pre else rmsecv_raw_asp
best_lv_asp = int(np.argmin(rmsecv_asp)) + 1

rmsecv_suc = rmsecv_grouped(X_cal, Y_suc_cal, g_cal, MAX_LV)
best_lv_suc = int(np.argmin(rmsecv_suc)) + 1
print(f"best LV(阿斯巴甜)={best_lv_asp}   best LV(三氯蔗糖)={best_lv_suc}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(range(1, MAX_LV + 1), rmsecv_raw_asp, "o-", label="raw")
ax.plot(range(1, MAX_LV + 1), rmsecv_pre_asp, "s-", label="baseline + area-normalize")
ax.axvline(best_lv_asp, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Number of latent variables (LV)")
ax.set_ylabel("RMSECV (%), aspartame")
ax.set_title("Preprocessing comparison (grouped LeaveOneGroupOut CV)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. 最終 PLS1 模型與盲樣 RMSEP

用選定的前處理與 LV 數，在整個校正集上配適最終模型，然後對 7 個離網格盲樣做**唯一一次**預測，計算 RMSEC、RMSECV、RMSEP、bias、RPD。

In [ ]:
blind_meta = meta[blind_mask].reset_index(drop=True)
X_blind_raw = X_all[blind_mask]
X_blind = preprocess_baseline_norm(X_blind_raw) if use_pre else X_blind_raw
Y_asp_blind = blind_meta["aspartame_pct"].to_numpy()
Y_suc_blind = blind_meta["sucralose_pct"].to_numpy()

pls_asp = PLSRegression(n_components=best_lv_asp).fit(X_cal, Y_asp_cal)
pls_suc = PLSRegression(n_components=best_lv_suc).fit(X_cal, Y_suc_cal)

logo = LeaveOneGroupOut()
cv_pred_asp = np.zeros_like(Y_asp_cal, dtype=float)
for tr, va in logo.split(X_cal, Y_asp_cal, g_cal):
    m = PLSRegression(n_components=best_lv_asp).fit(X_cal[tr], Y_asp_cal[tr])
    cv_pred_asp[va] = m.predict(X_cal[va]).ravel()
cv_pred_suc = np.zeros_like(Y_suc_cal, dtype=float)
for tr, va in logo.split(X_cal, Y_suc_cal, g_cal):
    m = PLSRegression(n_components=best_lv_suc).fit(X_cal[tr], Y_suc_cal[tr])
    cv_pred_suc[va] = m.predict(X_cal[va]).ravel()

pred_blind_asp = pls_asp.predict(X_blind).ravel()
pred_blind_suc = pls_suc.predict(X_blind).ravel()


def figures_of_merit(y_cal, model, X_cal_, cv_pred, y_blind, pred_blind):
    rmsec = float(np.sqrt(np.mean((model.predict(X_cal_).ravel() - y_cal) ** 2)))
    rmsecv = float(np.sqrt(np.mean((cv_pred - y_cal) ** 2)))
    rmsep = float(np.sqrt(np.mean((pred_blind - y_blind) ** 2)))
    bias = float(np.mean(pred_blind - y_blind))
    rpd = float(np.std(y_blind, ddof=1) / rmsep)
    return rmsec, rmsecv, rmsep, bias, rpd


rmsec_asp, rmsecv_asp_final, rmsep_asp, bias_asp, rpd_asp = figures_of_merit(
    Y_asp_cal, pls_asp, X_cal, cv_pred_asp, Y_asp_blind, pred_blind_asp)
rmsec_suc, rmsecv_suc_final, rmsep_suc, bias_suc, rpd_suc = figures_of_merit(
    Y_suc_cal, pls_suc, X_cal, cv_pred_suc, Y_suc_blind, pred_blind_suc)

fom = pd.DataFrame([
    {"sweetener": "aspartame", "LV": best_lv_asp, "RMSEC": rmsec_asp, "RMSECV": rmsecv_asp_final,
     "RMSEP": rmsep_asp, "bias": bias_asp, "RPD": rpd_asp},
    {"sweetener": "sucralose", "LV": best_lv_suc, "RMSEC": rmsec_suc, "RMSECV": rmsecv_suc_final,
     "RMSEP": rmsep_suc, "bias": bias_suc, "RPD": rpd_suc},
])
fom

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(Y_asp_cal, cv_pred_asp, s=40, color="tab:blue", alpha=0.7, label="Calibration (grouped CV)")
ax.scatter(Y_asp_blind, pred_blind_asp, s=80, color="tab:orange", marker="^", label="Blind blends")
lo, hi = -1, 11
ax.plot([lo, hi], [lo, hi], "--", color="grey", label="1:1 line")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Actual aspartame (%)")
ax.set_ylabel("PLS-predicted aspartame (%)")
ax.set_title("Aspartame PLS1: predicted vs actual")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 6. VIP vs Raman Shift

跟 NB12 用完全一樣的標準公式計算 VIP，對照官能基指認：阿斯巴甜的 VIP 應該在 **~1003 cm⁻¹**（苯環呼吸振動）附近最高；三氯蔗糖的 VIP 應該在 **600–850 cm⁻¹**（C–Cl 區）附近偏高。

In [ ]:
def compute_vip(pls):
    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_
    p, ncomp = W.shape
    SSY = np.array([(Q[0, a] ** 2) * np.sum(T[:, a] ** 2) for a in range(ncomp)])
    Wn = W / np.linalg.norm(W, axis=0, keepdims=True)
    return np.sqrt(p * (Wn ** 2 @ SSY) / SSY.sum())


vip_asp = compute_vip(pls_asp)
vip_suc = compute_vip(pls_suc)

print(f"mean(VIP_asp^2) = {np.mean(vip_asp**2):.9f}   mean(VIP_suc^2) = {np.mean(vip_suc**2):.9f}")
assert abs(np.mean(vip_asp ** 2) - 1.0) < 1e-9
assert abs(np.mean(vip_suc ** 2) - 1.0) < 1e-9

idx_1003 = int(np.argmin(np.abs(wn - 1003)))
peak_idx_asp = int(np.argmax(vip_asp))
print(f"VIP(阿斯巴甜) 全域最大值在 {wn[peak_idx_asp]:.0f} cm-1 (最接近1003的格點是 {wn[idx_1003]:.0f} cm-1)")
assert peak_idx_asp == idx_1003, "阿斯巴甜模型的最高 VIP 應該落在 1003 cm-1 附近"

c_cl_mask = (wn >= 600) & (wn <= 850)
print(f"VIP(三氯蔗糖) 在 600-850 cm-1 (C-Cl 區) 的最大值 = {vip_suc[c_cl_mask].max():.3f}")
assert vip_suc[c_cl_mask].max() > 1.5, "三氯蔗糖模型應該在 C-Cl 區有明顯高 VIP"
print("三個結構性檢查（mean(VIP^2)=1 x2、VIP高峰對上化學指認）全部通過。")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(wn, vip_asp, color="tab:blue")
axes[0].axhline(1.0, color="tab:red", linestyle="--", linewidth=1)
axes[0].axvline(1003, color="green", linestyle=":", linewidth=1.2, label="1003 cm-1 (aspartame ring breathing)")
axes[0].set_ylabel("VIP (aspartame model)")
axes[0].legend(fontsize=8)

axes[1].plot(wn, vip_suc, color="tab:purple")
axes[1].axhline(1.0, color="tab:red", linestyle="--", linewidth=1)
axes[1].axvspan(600, 850, color="orange", alpha=0.15, label="600-850 cm-1 (C-Cl region)")
axes[1].set_xlabel("Raman shift (cm-1)")
axes[1].set_ylabel("VIP (sucralose model)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

ery_bands = [880, 1050, 1290]
ery_idx = [int(np.argmin(np.abs(wn - w))) for w in ery_bands]
print("VIP(阿斯巴甜模型) 在赤藻糖醇填充劑波段 (880/1050/1290 cm-1):", np.round(vip_asp[ery_idx], 3))

**封閉性（closure）問題**：因為三個成分的比例加總永遠是 100%，赤藻糖醇的波段也可能出現 VIP > 1——阿斯巴甜或三氯蔗糖增加時，赤藻糖醇一定跟著減少，填充劑波段因此帶有「間接」的預測力。但這種間接關聯**不具特異性**：換一批用不同填充劑（例如市售代糖常見的糊精）的樣品，這個關聯就會失效，第 9 節會實際示範這個狀況。

## 7. NNLS／古典最小平方（不需要校正集的另一條路徑）

只用**純物質參考光譜**（`role == "library"` 的三筆光譜）做非負最小平方擬合：

$$\text{未知光譜} \approx c_1 \cdot \text{阿斯巴甜參考} + c_2 \cdot \text{三氯蔗糖參考} + c_3 \cdot \text{赤藻糖醇參考} + \text{offset}, \quad c_1, c_2, c_3 \ge 0$$

不需要另外準備校正集，$c$ 值直接就是重量分率，物理意義清楚；缺點是假設**線性疊加**，遇到偵測器飽和、基質效應等非線性時，比例估計會有系統性偏誤。

In [ ]:
lib_meta = meta[lib_mask].reset_index(drop=True)
lib_spec = {row.loc["class"]: X_all[lib_mask][i] for i, (_, row) in enumerate(lib_meta.iterrows())}
A_ref = np.column_stack([lib_spec["aspartame"], lib_spec["sucralose"], lib_spec["erythritol"], np.ones(len(wn))])


def nnls_fit(spec):
    coef, _ = nnls(A_ref, spec)
    return coef[0] * 100, coef[1] * 100, coef[2] * 100, coef[3]


nnls_cal = np.array([nnls_fit(x) for x in X_cal_raw])
nnls_blind = np.array([nnls_fit(x) for x in X_blind_raw])

rmsec_nnls_asp = float(np.sqrt(np.mean((nnls_cal[:, 0] - Y_asp_cal) ** 2)))
rmsep_nnls_asp = float(np.sqrt(np.mean((nnls_blind[:, 0] - Y_asp_blind) ** 2)))
rmsec_nnls_suc = float(np.sqrt(np.mean((nnls_cal[:, 1] - Y_suc_cal) ** 2)))
rmsep_nnls_suc = float(np.sqrt(np.mean((nnls_blind[:, 1] - Y_suc_blind) ** 2)))

method_compare = pd.DataFrame([
    {"method": f"PLS1 (LV={best_lv_asp}/{best_lv_suc}, {PREP_NAME})", "RMSEP_aspartame": rmsep_asp, "RMSEP_sucralose": rmsep_suc},
    {"method": "NNLS/CLS (3 components + offset)", "RMSEP_aspartame": rmsep_nnls_asp, "RMSEP_sucralose": rmsep_nnls_suc},
])
method_compare

PLS 在校正時已經「看過」偵測器輕微飽和等真實變異，通常 RMSEP 較小；NNLS 假設嚴格線性疊加，遇到這些變異時誤差較大——但換一批完全沒訓練過的新成分時（例如第 9 節的市售樣品），NNLS 不會亂外插，反而可能比 PLS「誠實」。

## 8. 近似 LOD（教學簡化法，非正式方法確效）

用交叉驗證預測值裡「0% 水準」的標準差，除以「預測值 vs 實際值」迴歸的斜率，估計一個近似的 LOD／LOQ：

$$\mathrm{LOD} \approx \dfrac{3.3\,\sigma_0}{\text{slope}}, \qquad \mathrm{LOQ} \approx \dfrac{10\,\sigma_0}{\text{slope}}$$

> ⚠️ 這是**教學上的簡化估計法**，不是正式的方法確效（method validation）流程（正式做法需要多次獨立重複量測空白/低濃度樣品，並考慮更嚴謹的統計假設）。

In [ ]:
def approx_lod(y_design, cv_pred):
    mask0 = (y_design == 0.0)
    sd0 = float(np.std(cv_pred[mask0], ddof=1))
    slope, intercept = np.polyfit(y_design, cv_pred, 1)
    lod = 3.3 * sd0 / slope
    loq = 10 * sd0 / slope
    return lod, loq, sd0, slope


LOD_asp, LOQ_asp, sd0_asp, slope_asp = approx_lod(Y_asp_cal, cv_pred_asp)
LOD_suc, LOQ_suc, sd0_suc, slope_suc = approx_lod(Y_suc_cal, cv_pred_suc)

print(f"阿斯巴甜: slope={slope_asp:.4f}  sd(0%)={sd0_asp:.4f}  LOD≈{LOD_asp:.3f}%  LOQ≈{LOQ_asp:.3f}%")
print(f"三氯蔗糖  : slope={slope_suc:.4f}  sd(0%)={sd0_suc:.4f}  LOD≈{LOD_suc:.3f}%  LOQ≈{LOQ_suc:.3f}%")

## 9. 市售樣品的適用範圍檢查（T²／Q）

跟 NB12 用一樣的方法（PCA + Hotelling T² + Jackson–Mudholkar Q 界限）檢查一個「市售代糖」樣品是否落在模型的**適用範圍**內。這包代糖標示阿斯巴甜 1.8%，但填充劑是**糊精**（dextrose-like），跟校正集用的**赤藻糖醇**完全不同。

In [ ]:
commercial_asp_true = float(meta.loc[unk_mask, "aspartame_pct"].iloc[0])  # declared label value

pca_full = PCA(n_components=min(len(X_cal) - 1, X_cal.shape[1] - 1)).fit(X_cal)
eigvals_all = pca_full.explained_variance_

NCOMP_PCA = 3
pca = PCA(n_components=NCOMP_PCA).fit(X_cal)
eigvals = pca.explained_variance_[:NCOMP_PCA]

n_cal_pts = X_cal.shape[0]
F_crit = stats.f.ppf(0.95, NCOMP_PCA, n_cal_pts - NCOMP_PCA)
T2_limit = NCOMP_PCA * (n_cal_pts - 1) / (n_cal_pts - NCOMP_PCA) * F_crit

theta1 = float(np.sum(eigvals_all[NCOMP_PCA:]))
theta2 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 2))
theta3 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 3))
h0 = 1 - (2 * theta1 * theta3) / (3 * theta2 ** 2)
z95 = stats.norm.ppf(0.95)
Q_limit = theta1 * (z95 * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)
print(f"T2_limit (95%) = {T2_limit:.3f}   Q_limit (95%) = {Q_limit:.6f}")

comm_raw = X_all[unk_mask][0]
comm_x = preprocess_baseline_norm(comm_raw.reshape(1, -1))[0] if use_pre else comm_raw

pred_comm_asp = float(np.ravel(pls_asp.predict(comm_x.reshape(1, -1)))[0])
T_comm = pca.transform(comm_x.reshape(1, -1))
T2_comm = float(np.sum((T_comm ** 2) / eigvals))
recon_comm = pca.inverse_transform(T_comm)
Q_comm = float(np.sum((comm_x - recon_comm) ** 2))

print(f"\n標示阿斯巴甜 = {commercial_asp_true}%   PLS 直接預測 = {pred_comm_asp:.2f}%")
print(f"T2={T2_comm:.2f} (界限{T2_limit:.2f}, {'超出' if T2_comm > T2_limit else '正常'})   "
      f"Q={Q_comm:.5f} (界限{Q_limit:.5f}, {'超出' if Q_comm > Q_limit else '正常'})")

assert Q_comm > Q_limit, "市售樣品使用了校正集沒見過的填充劑，Q 殘差應明顯超標"
print("\n檢查通過：Q 殘差超標，代表這包代糖『長得不像』校正集學過的任何東西——"
      "PLS 的預測值不能直接採信，即使數字本身看起來『合理』。")

**誠實的報告方式**：先看 Q／T²，超出適用範圍就不能直接報數字，只能寫「超出模型適用範圍，建議重新採集校正集或改用其他方法」；若 Q／T² 正常但預測值低於 LOD，則報告「< LOD」，而不是硬報一個看起來精確、實際上沒有意義的數字。

## 10. W17 模型報告卡

In [ ]:
report_card = pd.DataFrame([
    {"item": "儀器", "value": "sim_Raman785_demo"},
    {"item": "前處理", "value": PREP_NAME},
    {"item": "LV (阿斯巴甜/三氯蔗糖)", "value": f"{best_lv_asp}/{best_lv_suc}"},
    {"item": "RMSEC/RMSECV/RMSEP (阿斯巴甜, %)", "value": f"{rmsec_asp:.2f}/{rmsecv_asp_final:.2f}/{rmsep_asp:.2f}"},
    {"item": "RMSEC/RMSECV/RMSEP (三氯蔗糖, %)", "value": f"{rmsec_suc:.2f}/{rmsecv_suc_final:.2f}/{rmsep_suc:.2f}"},
    {"item": "bias/RPD (阿斯巴甜)", "value": f"{bias_asp:+.2f}% / {rpd_asp:.2f}"},
    {"item": "LOD (阿斯巴甜/三氯蔗糖)", "value": f"{LOD_asp:.2f}% / {LOD_suc:.2f}%"},
    {"item": "VIP-波段吻合度", "value": "阿斯巴甜峰值在1003cm-1(符合)；三氯蔗糖C-Cl區VIP>1.5(符合)"},
    {"item": "NNLS 比較", "value": f"PLS較優 ({rmsep_asp:.2f} vs {rmsep_nnls_asp:.2f}% 阿斯巴甜)"},
    {"item": "商用樣品結果", "value": "超出適用範圍 (Q超標)，未報數值" if Q_comm > Q_limit else "在適用範圍內"},
])
report_card

## 11. 練習題 (Exercises)

### 問題 1

把第 2 節的正交性檢查改成故意打亂三氯蔗糖那一欄的順序（例如 `np.random.permutation`）後再算一次 `corr()`。相關係數還會接近 0 嗎？這個練習想說明什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 2（資料外洩示範）

把第 4 節的 `LeaveOneGroupOut`（以 `sample_id` 分組）換成不分組的普通 `KFold`，對阿斯巴甜模型重新算一次 RMSECV（固定用 `best_lv_asp`）。跟分組結果比較，哪一個比較樂觀？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 3

對照第 6 節的 VIP 圖與封閉性（closure）說明：如果今天赤藻糖醇換成一個化學結構跟阿斯巴甜、三氯蔗糖完全不重疊的填充劑，你預期赤藻糖醇波段的 VIP 會比現在更高、更低，還是差不多？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

第 7 節 NNLS 對盲樣的 RMSEP 比 PLS 差。試著解釋：如果今天完全沒有校正集（沒有 25 個已知配方），只有三條純物質參考光譜，你還能用 PLS 嗎？為什麼 NNLS 在這種情況下是唯一可行的選擇？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

第 9 節的市售樣品，PLS 直接預測出一個數字，但 Q 殘差超標。如果今天 Q／T² 都正常、但預測值低於第 8 節算出的 LOD，你應該在報告中怎麼寫這個樣品的分析結果？

In [ ]:
# TODO: 請在這裡作答

## 12. 匯出結果 (Export)

In [ ]:
output_path = "nb13_model_report_card.csv"
report_card.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")